In [2]:
#Demarrage de Spark , creation de session

In [3]:

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("MedicalLakehouse")
    .config(
        "spark.sql.extensions",
        "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions"
    )
    .config(
        "spark.sql.catalog.medical",
        "org.apache.iceberg.spark.SparkCatalog"
    )
    .config("spark.sql.catalog.medical.type", "rest")
    .config(
        "spark.sql.catalog.medical.uri",
        "http://iceberg-rest:8181"
    )
    .config(
        "spark.sql.catalog.medical.warehouse",
        "s3://medical-warehouse/"
    )
    .config(
        "spark.sql.catalog.medical.io-impl",
        "org.apache.iceberg.aws.s3.S3FileIO"
    )
    .config(
        "spark.sql.catalog.medical.s3.endpoint",
        "http://rustfs:9000"
    )
    .config(
        "spark.sql.catalog.medical.s3.path-style-access",
        "true"
    )
    .getOrCreate()
)

print("Spark :", spark.version)


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/19 00:24:55 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/19 00:24:55 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark : 3.5.5


In [ ]:
#Lire patients.csv directement avec Spark

In [4]:
chemin = "/workspace/notebooks/data/synthea/patients.csv"

df_patients = spark.read.csv(
    chemin,
    header=True,
    inferSchema=False
)

df_patients.printSchema()
df_patients.show(5, truncate=False)

root
 |-- Id: string (nullable = true)
 |-- BIRTHDATE: string (nullable = true)
 |-- DEATHDATE: string (nullable = true)
 |-- SSN: string (nullable = true)
 |-- DRIVERS: string (nullable = true)
 |-- PASSPORT: string (nullable = true)
 |-- PREFIX: string (nullable = true)
 |-- FIRST: string (nullable = true)
 |-- MIDDLE: string (nullable = true)
 |-- LAST: string (nullable = true)
 |-- SUFFIX: string (nullable = true)
 |-- MAIDEN: string (nullable = true)
 |-- MARITAL: string (nullable = true)
 |-- RACE: string (nullable = true)
 |-- ETHNICITY: string (nullable = true)
 |-- GENDER: string (nullable = true)
 |-- BIRTHPLACE: string (nullable = true)
 |-- ADDRESS: string (nullable = true)
 |-- CITY: string (nullable = true)
 |-- STATE: string (nullable = true)
 |-- COUNTY: string (nullable = true)
 |-- FIPS: string (nullable = true)
 |-- ZIP: string (nullable = true)
 |-- LAT: string (nullable = true)
 |-- LON: string (nullable = true)
 |-- HEALTHCARE_EXPENSES: string (nullable = true)
 |

26/10/19 00:25:24 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------------------------------+----------+---------+-----------+---------+----------+------+------------+----------+-------------+------+-----------+-------+-----+-----------+------+-----------------------------+--------------------------+----------------+-------------+----------------+-----+-----+-----------------+------------------+-------------------+-------------------+------+
|Id                                  |BIRTHDATE |DEATHDATE|SSN        |DRIVERS  |PASSPORT  |PREFIX|FIRST       |MIDDLE    |LAST         |SUFFIX|MAIDEN     |MARITAL|RACE |ETHNICITY  |GENDER|BIRTHPLACE                   |ADDRESS                   |CITY            |STATE        |COUNTY          |FIPS |ZIP  |LAT              |LON               |HEALTHCARE_EXPENSES|HEALTHCARE_COVERAGE|INCOME|
+------------------------------------+----------+---------+-----------+---------+----------+------+------------+----------+-------------+------+-----------+-------+-----+-----------+------+-----------------------------

In [ ]:
#Sélectionner les quatre colonnes nécessaires

In [5]:
df_patients_source = df_patients.select(
    F.col("Id").alias("patient_id"),
    F.col("BIRTHDATE").alias("birthdate"),
    F.col("GENDER").alias("gender"),
    F.col("CITY").alias("city")
)

df_patients_source.show(5, truncate=False)

print("Nombre de patients :", df_patients_source.count())

print(
    "Nombre d'identifiants distincts :",
    df_patients_source.select("patient_id").distinct().count()
)

+------------------------------------+----------+------+----------------+
|patient_id                          |birthdate |gender|city            |
+------------------------------------+----------+------+----------------+
|cc3eac4a-34a7-9a15-1522-8087135631b7|1978-01-16|M     |North Brookfield|
|8b9df7af-24b3-66ee-afeb-2734ebcbd33c|2001-02-11|F     |Bedford         |
|65cbad6a-eb9d-420e-9bab-b4f9c6aab7db|1948-06-15|F     |Quincy          |
|9a3814bb-f5b0-05f3-928e-fe6640d5aa2a|2015-07-19|F     |Taunton         |
|6a703775-432b-ba7d-d53a-c4d7bb0f5b58|1995-01-23|M     |Boston          |
+------------------------------------+----------+------+----------------+
only showing top 5 rows

Nombre de patients : 108
Nombre d'identifiants distincts : 108


In [ ]:
#Envoyer les patients dans PostgreSQL

In [14]:
jdbc_url = "jdbc:postgresql://postgres:5432/hospital"

jdbc_options = {
    "url": jdbc_url,
    "dbtable": "patients_source",
    "user": "user",
    "password": "0000",
    "driver": "org.postgresql.Driver"
}

(
    df_patients_source.write
    .format("jdbc")
    .options(**jdbc_options)
    .mode("errorifexists")
    .save()
)

print("Patients importés dans PostgreSQL !")

Patients importés dans PostgreSQL !


In [15]:
df_verification = (
    spark.read
    .format("jdbc")
    .options(**jdbc_options)
    .load()
)

df_verification.show(5, truncate=False)

print(
    "Patients dans PostgreSQL :",
    df_verification.count()
)

+------------------------------------+----------+------+----------------+
|patient_id                          |birthdate |gender|city            |
+------------------------------------+----------+------+----------------+
|cc3eac4a-34a7-9a15-1522-8087135631b7|1978-01-16|M     |North Brookfield|
|8b9df7af-24b3-66ee-afeb-2734ebcbd33c|2001-02-11|F     |Bedford         |
|65cbad6a-eb9d-420e-9bab-b4f9c6aab7db|1948-06-15|F     |Quincy          |
|9a3814bb-f5b0-05f3-928e-fe6640d5aa2a|2015-07-19|F     |Taunton         |
|6a703775-432b-ba7d-d53a-c4d7bb0f5b58|1995-01-23|M     |Boston          |
+------------------------------------+----------+------+----------------+
only showing top 5 rows

Patients dans PostgreSQL : 108


In [ ]:
#Étape 0.4 — Préparer les observations au format JSON
#Convertir les observations CSV en JSON

In [17]:

import csv
import json
from pathlib import Path

source = Path(
    "/workspace/notebooks/data/synthea/observations.csv"
)

destination = Path(
    "/workspace/notebooks/data/prepared/observations.jsonl"
)

destination.parent.mkdir(parents=True, exist_ok=True)

nombre_observations = 0

with source.open("r", encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f)

    with destination.open("w", encoding="utf-8") as out:
        for observation in reader:
            out.write(
                json.dumps(observation, ensure_ascii=False) + "\n"
            )
            nombre_observations += 1

print("Conversion terminée !")
print("Nombre d'observations :", nombre_observations)
print("Fichier créé :", destination)
df_observations = spark.read.json(
    "/workspace/notebooks/data/prepared/observations.jsonl"
)

df_observations.printSchema()

df_observations.show(5, truncate=False)

print("Observations lues :", df_observations.count())

Conversion terminée !
Nombre d'observations : 68648
Fichier créé : /workspace/notebooks/data/prepared/observations.jsonl


root
 |-- CATEGORY: string (nullable = true)
 |-- CODE: string (nullable = true)
 |-- DATE: string (nullable = true)
 |-- DESCRIPTION: string (nullable = true)
 |-- ENCOUNTER: string (nullable = true)
 |-- PATIENT: string (nullable = true)
 |-- TYPE: string (nullable = true)
 |-- UNITS: string (nullable = true)
 |-- VALUE: string (nullable = true)

+-----------+-------+--------------------+-------------------------------------------------------------+------------------------------------+------------------------------------+-------+-------+-----+
|CATEGORY   |CODE   |DATE                |DESCRIPTION                                                  |ENCOUNTER                           |PATIENT                             |TYPE   |UNITS  |VALUE|
+-----------+-------+--------------------+-------------------------------------------------------------+------------------------------------+------------------------------------+-------+-------+-----+
|laboratory |4548-4 |2018-01-22T08:35:36Z|Hemo

In [18]:
#Charger les consultations CSV avec Spark

In [19]:
df_consultations = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(
        "/workspace/notebooks/data/synthea/encounters.csv"
    )
)

df_consultations.printSchema()

df_consultations.show(5, truncate=False)

print("Nombre de consultations :", df_consultations.count())

root
 |-- Id: string (nullable = true)
 |-- START: string (nullable = true)
 |-- STOP: string (nullable = true)
 |-- PATIENT: string (nullable = true)
 |-- ORGANIZATION: string (nullable = true)
 |-- PROVIDER: string (nullable = true)
 |-- PAYER: string (nullable = true)
 |-- ENCOUNTERCLASS: string (nullable = true)
 |-- CODE: string (nullable = true)
 |-- DESCRIPTION: string (nullable = true)
 |-- BASE_ENCOUNTER_COST: string (nullable = true)
 |-- TOTAL_CLAIM_COST: string (nullable = true)
 |-- PAYER_COVERAGE: string (nullable = true)
 |-- REASONCODE: string (nullable = true)
 |-- REASONDESCRIPTION: string (nullable = true)

+------------------------------------+--------------------+--------------------+------------------------------------+------------------------------------+------------------------------------+------------------------------------+--------------+---------+------------------------------------------+-------------------+----------------+--------------+----------+-------

In [ ]:
###Récapitulatif des données chargées

In [20]:
print("=== SOURCES MÉDICALES ===")

print("Patients PostgreSQL :", df_verification.count())

print("Consultations CSV :", df_consultations.count())

print("Observations JSON :", df_observations.count())

=== SOURCES MÉDICALES ===
Patients PostgreSQL : 108
Consultations CSV : 5571
Observations JSON : 68648
